In [3]:
# [Cell 1]
# Databricks Free Edition: Spark runs on serverless via Databricks Connect (see dldg.py at the repo root)
from dldg import spark, DATASETS, WORK, set_conf, tree, rm, upload_dir
CH = f"{WORK}/ch05"  # replaces the Docker image's /opt/spark/work-dir

In [4]:
# [Cell 2]
# run first for imports
from pyspark.sql.types import DateType
from pyspark.sql.functions import col, desc
from delta.tables import DeltaTable

*[Cell 3]*

# Chapter 5: Maintaining your Delta Lake (Extras)
> The following exercises use the open-source [nyc_taxi dataset](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page), specifically the Yellow Taxi Trip Records parquet. 

In [5]:
# [Cell 4]
spark.sql("""
     CREATE TABLE IF NOT EXISTS workspace.dldg.nyc_taxi (
       VendorID BIGINT
     ) USING DELTA
     TBLPROPERTIES(
       'delta.logRetentionDuration'='interval 7 days',
       -- the taxi data has TIMESTAMP_NTZ columns, which Databricks requires you to opt in to
       'delta.feature.timestampNtz'='supported'
     );
   """)


DataFrame[]

In [6]:
# [Cell 5]
spark.sql("show tables").show()

+--------+---------+-----------+
|database|tableName|isTemporary|
+--------+---------+-----------+
|    dldg| nyc_taxi|      false|
+--------+---------+-----------+



In [7]:
# [Cell 6]
# spark.sql("drop table workspace.dldg.nyc_taxi")

In [8]:
# [Cell 7]
# should be zero on your first pass
# this means that while the table exists, there are no data files associated with it
len(spark.table("workspace.dldg.nyc_taxi").inputFiles())

0

*[Cell 8]*

## Start Populating the Table
> The next three commands are used to show Schema Evolution and Validation with Delta Lake


In [9]:
# [Cell 9]
# Populate the Table reading the Parquet nyc_taxi Data
# note: this will fail and that is okay
(spark.read
      .format("parquet")
      .load(f"{DATASETS}/nyc_taxi/yellow_tripdata_2023-01.parquet")
      .write
      .format("delta")
      .saveAsTable("workspace.dldg.nyc_taxi"))


AnalysisException: [TABLE_OR_VIEW_ALREADY_EXISTS] Cannot create table or view `dldg`.`nyc_taxi` because it already exists.
Choose a different name, drop the existing object, add the IF NOT EXISTS clause to tolerate pre-existing objects, add the OR REPLACE clause to replace the existing materialized view, or add the OR REFRESH clause to refresh the existing streaming table. SQLSTATE: 42P07

JVM stacktrace:
org.apache.spark.sql.catalyst.analysis.TableAlreadyExistsException
	at org.apache.spark.sql.errors.QueryCompilationErrors$.tableAlreadyExistsError(QueryCompilationErrors.scala:3379)
	at org.apache.spark.sql.execution.datasources.v2.AtomicCreateTableAsSelectExec.run(WriteToDataSourceV2Exec.scala:225)
	at org.apache.spark.sql.execution.datasources.v2.V2CommandExec.$anonfun$result$3(V2CommandExec.scala:55)
	at org.apache.spark.sql.execution.SparkPlan.runCommandInAetherOrSpark(SparkPlan.scala:231)
	at org.apache.spark.sql.execution.datasources.v2.V2CommandExec.$anonfun$result$2(V2CommandE

In [10]:
# [Cell 10]
# one step closer, there is still something missing...
# and yes, this operation still fails... if only...
(spark.read
      .format("parquet")
      .load(f"{DATASETS}/nyc_taxi/yellow_tripdata_2023-01.parquet")
      .write
      .format("delta")
      .mode("append")
      .saveAsTable("workspace.dldg.nyc_taxi"))

AnalysisException: [DELTA_METADATA_MISMATCH] A metadata mismatch was detected when writing to the Delta table.
- A schema mismatch detected when writing to the Delta table (Table ID: 2aadf63e-ebe3-4c0c-9158-8b92d7054d9c).
To enable schema migration using DataFrameWriter or DataStreamWriter, please set: '.option("mergeSchema", "true")'.
For other operations, set the session configuration spark.databricks.delta.schema.autoMerge.enabled to "true". See the documentation specific to the operation for details.

Table schema:
root
 |-- VendorID: long (nullable = true)


Data schema:
root
 |-- VendorID: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (null

*[Cell 11]*

## Schema Evolution: Handle Automatically
If you trust the upstream data source (provider) then you can add the `option("mergeSchema", "true")`. Otherwise, it is better to specifically select a subset of the columns you expected to see. In this example use case, the only known column is the `VendorID`.

In [11]:
# [Cell 12]
# Evolve the Schema. (Showcases how to auto-merge changes to the schema)
# note: if you can trust the upstream, then this option is perfectly fine
# however, if you don't trust the upstream, then it is good to opt-in to the 
# changing columns.

(spark.read
      .format("parquet")
      .load(f"{DATASETS}/nyc_taxi/yellow_tripdata_2023-01.parquet")
      .write
      .format("delta")
      .mode("append")
      .option("mergeSchema", "true")
      .saveAsTable("workspace.dldg.nyc_taxi")
    )


*[Cell 13]*

# Alternatives to Auto Schema Evolution
In the previous case, we used `.option("mergeSchema", "true")` to modify the behavior of the Delta Lake writer. While this option simplifies how we evolve our Delta Lake table schemas, it comes at the price of not being fully aware of the changes to our table schema. In the case where there are unknown columns being introduced from an upstream source, you'll want to know which columns are intended to bring forward, and which columns can be safely ignored.

## Intentionally Adding Columns with Alter Table

In [12]:
# [Cell 14]
# manually set the columns. This is an example of intentional opt-in to the new columns outside of '.option("mergeSchema", "true")`. 
# Note: ADD columns fails if a column already exists (FIELD_ALREADY_EXISTS). The mergeSchema write above
# already added them, so only add the ones still missing; on a fresh table this runs the full ALTER TABLE.
new_columns = {
    "tpep_pickup_datetime": "TIMESTAMP",
    "tpep_dropoff_datetime": "TIMESTAMP",
    "passenger_count": "DOUBLE",
    "trip_distance": "DOUBLE",
    "RatecodeID": "DOUBLE",
    "store_and_fwd_flag": "STRING",
    "PULocationID": "BIGINT",
    "DOLocationID": "BIGINT",
    "payment_type": "BIGINT",
    "fare_amount": "DOUBLE",
    "extra": "DOUBLE",
    "mta_tax": "DOUBLE",
    "tip_amount": "DOUBLE",
    "tolls_amount": "DOUBLE",
    "improvement_surcharge": "DOUBLE",
    "total_amount": "DOUBLE",
    "congestion_surcharge": "DOUBLE",
    "airport_fee": "DOUBLE"
}
missing = [f"{name} {dtype}" for name, dtype in new_columns.items()
           if name not in spark.table("workspace.dldg.nyc_taxi").columns]
if missing:
    spark.sql(f"""
    ALTER TABLE workspace.dldg.nyc_taxi 
    ADD columns (
      {", ".join(missing)}
    );
    """)
else:
    print("Columns already exist (added by mergeSchema above); skipping ALTER TABLE")


Columns already exist (added by mergeSchema above); skipping ALTER TABLE


In [14]:
# [Cell 15]
# view the table structure using DESCRIBE
spark.sql("describe extended workspace.dldg.nyc_taxi").show(30, truncate=False)

+----------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------+
|col_name                    |data_type                                                                                                                                                                                                                                                                                     |comment|
+----------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------+
|VendorID             

In [ ]:
# [Cell 16]
spark.sql("select * from workspace.dldg.nyc_taxi limit 10").show(truncate=True)

*[Cell 17]*

# Adding and Modifying Table Properties

In [15]:
# [Cell 18]
spark.sql("""
ALTER TABLE workspace.dldg.nyc_taxi 
SET TBLPROPERTIES (
  -- Databricks requires logRetentionDuration >= deletedFileRetentionDuration (the book used 14 days here)
  'delta.logRetentionDuration'='interval 30 days',
  'delta.deletedFileRetentionDuration'='interval 28 days'
)""")


DataFrame[]

In [16]:
# [Cell 19]
# view the Table History
# this will show the SET TBLPROPERTIES transaction in the Delta Lake transaction log
dt = DeltaTable.forName(spark, 'workspace.dldg.nyc_taxi')
(
    dt
    .history(10)
    .select("version", "timestamp", "operation")
    .show(truncate=False)
)


+-------+-------------------+-----------------+
|version|timestamp          |operation        |
+-------+-------------------+-----------------+
|2      |2026-10-24 22:47:04|SET TBLPROPERTIES|
|1      |2026-10-24 22:28:54|WRITE            |
|0      |2026-10-24 21:51:09|CREATE TABLE     |
+-------+-------------------+-----------------+



In [17]:
# [Cell 20]
# view your Delta Lake tblproperties using Spark SQL
# you'll see the tblproperties we added as well as the delta.minReaderVersion, delta.minWriterVersion
spark.sql("show tblproperties workspace.dldg.nyc_taxi").show(truncate=False)

+--------------------------------------------------------+------------------------------------------------------------+
|key                                                     |value                                                       |
+--------------------------------------------------------+------------------------------------------------------------+
|delta.checkpointPolicy                                  |v2                                                          |
|delta.deletedFileRetentionDuration                      |interval 28 days                                            |
|delta.enableDeletionVectors                             |true                                                        |
|delta.enableRowTracking                                 |true                                                        |
|delta.feature.appendOnly                                |supported                                                   |
|delta.feature.deletionVectors          

In [18]:
# [Cell 21]
# viewing the properties using the DeltaTable command
(
    DeltaTable
    .forName(spark, 'workspace.dldg.nyc_taxi')
    .detail()
    .select("properties")
    .show(truncate=False)
)

+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|properties                                                                                                                                                                                                                                                                                                                                                                          

*[Cell 22]*

# Removing Table Properties

In [19]:
# [Cell 23]
# using unset to remove a property.
# in the case of a key not existing, the operation becomes noop
# so there is no need to use IF EXISTS conditionals
spark.sql("""
    ALTER TABLE workspace.dldg.nyc_taxi
    UNSET TBLPROPERTIES('delta.loRgetentionDuratio')
  """)


DataFrame[]

*[Cell 24]*

# Delta Table Optimizations (Cleaning, Tuning)
> The following section showcases how to create and fix a poorly optimized Delta Lake table
* Long Running Code will have a warning before hand.

In [20]:
# [Cell 25]
# create a new table called `nonoptimal_nyc_taxi` in the `default` database.
(DeltaTable.createIfNotExists(spark)
    .tableName("workspace.dldg.nonoptimal_nyc_taxi")
    .property("description", "table to be optimized")
    .addColumn("VendorID", "BIGINT")
    .addColumn("tpep_pickup_datetime", "TIMESTAMP")
    .addColumn("tpep_dropoff_datetime", "TIMESTAMP")
    .addColumn("passenger_count", "DOUBLE")
    .addColumn("trip_distance", "DOUBLE")
    .addColumn("RatecodeID", "DOUBLE")
    .addColumn("store_and_fwd_flag", "STRING")
    .addColumn("PULocationID", "BIGINT")
    .addColumn("DOLocationID", "BIGINT")
    .addColumn("payment_type", "BIGINT")
    .addColumn("fare_amount", "DOUBLE")
    .addColumn("extra", "DOUBLE")
    .addColumn("mta_tax", "DOUBLE")
    .addColumn("tip_amount", "DOUBLE")
    .addColumn("tolls_amount", "DOUBLE")
    .addColumn("improvement_surcharge", "DOUBLE")
    .addColumn("total_amount", "DOUBLE")
    .addColumn("congestion_surcharge", "DOUBLE")
    .addColumn("airport_fee", "DOUBLE")
    .execute()
  )


In [21]:
# [Cell 26]
spark.sql("show tables").show()

+--------+-------------------+-----------+
|database|          tableName|isTemporary|
+--------+-------------------+-----------+
|    dldg|nonoptimal_nyc_taxi|      false|
|    dldg|           nyc_taxi|      false|
+--------+-------------------+-----------+



In [23]:
# [Cell 27]
# Create a function to take a Row and save as a separate Table
# note: this is for example, this is not optimal
# note 2: if you want to use something similar, please use `rows` to write a collection of rows per transaction
def append_row_to_table(row, schema, table):
    (spark.createDataFrame([row], schema)
      .write
      .format("delta")
      .mode("append")
      .saveAsTable(table))


In [24]:
# [Cell 28]
####### Warning: will run for a while
source_df = spark.table("workspace.dldg.nyc_taxi")
source_schema = source_df.schema
destination_table = "workspace.dldg.nonoptimal_nyc_taxi"
# change limit to 1000 or 10000 if you want to generate a more verbose example
limit = 100

# warning: list comprehension is used here to run synchronous inserts and to prove a point
# please don't use this code for production use cases, and just to create a poorly optimized table
([
    append_row_to_table(row, source_schema, destination_table) 
    for row in source_df.limit(limit).collect()
])


[None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None]

*[Cell 29]*

## Using Optimize
> Using Bin-Packing Optimize (default) will allow us to coalesce many small files (which we just created) into fewer large files.

Spark Config:
1. `spark.databricks.delta.optimize.repartition.enabled=true` is useful when we have many small files like we do in the case of the nonoptimal_nyc_taxi Delta Lake Table.

Databricks-Only: Delta Lake Table Properties:
1. `delta.targetFileSize=20mb`
2. `delta.tuneFileSizesForRewrites=true`

When running OPTIMIZE outside of databricks, like we are inside this jupyter notebook, we can lean on some alternative spark configuration to control how many files we read and how we can optimize differently.

In [25]:
# [Cell 30]
# modify the table properties for OPTIMIZE

# note: these configurations are only for databricks at the time of writing so the 
# `spark.databricks.delta.allowArbitraryProperties.enabled` is used to prevent an exception from being thrown
set_conf('spark.databricks.delta.allowArbitraryProperties.enabled','true')
set_conf('spark.databricks.delta.optimize.repartition.enabled', 'true')
set_conf('spark.sql.files.maxRecordsPerFile', '1000000')
spark.sql("""
    ALTER TABLE workspace.dldg.nonoptimal_nyc_taxi
    SET TBLPROPERTIES (
      'delta.targetFileSize'='20mb',
      'delta.tuneFileSizesForRewrites'='true'
    )
  """)

Skipped spark.databricks.delta.allowArbitraryProperties.enabled=true: not settable on serverless compute
Skipped spark.databricks.delta.optimize.repartition.enabled=true: not settable on serverless compute
Skipped spark.sql.files.maxRecordsPerFile=1000000: not settable on serverless compute


DataFrame[]

In [26]:
# [Cell 31]
# execute OPTIMIZE from the Delta python client
df = (
    DeltaTable.forName(spark, "workspace.dldg.nonoptimal_nyc_taxi")
    .optimize()
    .executeCompaction()
)
df.show(truncate=True)

+----+--------------------+
|path|             metrics|
+----+--------------------+
|    |{NULL, NULL, NULL...|
+----+--------------------+



In [27]:
# [Cell 32]
# view the full metadata from the OPTIMIZE operation

(
    DeltaTable
    .forName(spark, "workspace.dldg.nonoptimal_nyc_taxi")
    .history(2)
    .where(col("operation") == "OPTIMIZE")
    .select("version", "timestamp", "operation", "operationMetrics.numRemovedFiles", "operationMetrics.numAddedFiles")
    .show(truncate=False)
)


+-------+-------------------+---------+---------------+-------------+
|version|timestamp          |operation|numRemovedFiles|numAddedFiles|
+-------+-------------------+---------+---------------+-------------+
|105    |2026-10-24 23:18:18|OPTIMIZE |7              |1            |
+-------+-------------------+---------+---------------+-------------+



*[Cell 33]*

# Using Z-Order Optimize
> Z-Order optimize is a co-location technique to minimize the total number of files loaded to answer common questions (queries) from your Delta Lake tables. For example, let’s say 80% of the queries to the `nyc_taxi dataset` always search first for `tpep_pickup_datetime` followed by a specific `RatecodeID`. You could optimize for faster query results by co-locating the `tpep_pickup_datetime` and `RatecodeID` so that the search space is reduced
> This allows us to reduce the number of files that need to be opened, using data skipping, since Delta Lake captures statistics automatically for the first 32 columns of a Delta Lake table.

## Delta Lake Table Properties
`delta.dataSkippingNumIndexedCols=6` could be used in the case where we only care about the first 6 columns of our Delta Lake table.


In [28]:
# [Cell 34]
# change the statistics collected for the table from the default 32 down to 6
# since we will be calling zorder optimize, this setting can take effect
# along with the operation itself
spark.sql("""
    ALTER TABLE workspace.dldg.nonoptimal_nyc_taxi
    SET TBLPROPERTIES (
      'delta.dataSkippingNumIndexedCols'='6'
    )
  """)

DataFrame[]

In [29]:
# [Cell 35]
# see https://docs.databricks.com/delta/data-skipping.html for more details
dt = DeltaTable.forName(spark, "workspace.dldg.nonoptimal_nyc_taxi")
(
    dt
    .optimize()
    .executeZOrderBy("tpep_pickup_datetime", "RatecodeID")
)


DataFrame[path: string, metrics: struct<autoCompactParallelismStats:void,clusteringMetrics:void,clusteringStats:void,deletionVectorStats:struct<numDeletionVectorRowsRemoved:bigint,numDeletionVectorsRemoved:bigint>,endTimeMs:bigint,fileAgeHistogram:void,filesAdded:struct<avg:double,max:void,min:void,totalFiles:bigint,totalSize:bigint>,filesRemoved:struct<avg:double,max:void,min:void,totalFiles:bigint,totalSize:bigint>,numBatches:bigint,numBins:bigint,numBytesSkippedToReduceWriteAmplification:bigint,numFilesAdded:bigint,numFilesRemoved:bigint,numFilesSkippedToReduceWriteAmplification:bigint,numFilesSortedWithinFiles:bigint,numTableColumns:bigint,numTableColumnsWithStats:bigint,partitionsOptimized:bigint,preserveInsertionOrder:boolean,recompressionCodec:void,skippedArchivedFiles:bigint,startTimeMs:bigint,totalClusterParallelism:bigint,totalConsideredFiles:bigint,totalFilesSkipped:bigint,totalScheduledTasks:bigint,totalTaskExecutionTimeMs:bigint,zOrderStats:struct<inputCubeFiles:struct<num

In [30]:
# [Cell 36]
# view the results of the optimization
# in the case where we user z-order optimize on a single file, it isn't going to help much
# but you get the idea!
(
    dt.history(10)
    .where(col("operation") == "OPTIMIZE")
    .select("version", "timestamp", "operation", "operationMetrics.numRemovedFiles", "operationMetrics.numAddedFiles")
    .show(truncate=False)
)

+-------+-------------------+---------+---------------+-------------+
|version|timestamp          |operation|numRemovedFiles|numAddedFiles|
+-------+-------------------+---------+---------------+-------------+
|105    |2026-10-24 23:18:18|OPTIMIZE |7              |1            |
|98     |2026-10-24 23:15:58|OPTIMIZE |32             |1            |
+-------+-------------------+---------+---------------+-------------+



*[Cell 37]*

# Partition Tuning
> Note: for tables under 1TB it isn't advised to use any partitioning and lean on OPTIMIZE and Z-ORDER OPTIMIZE.
> Edge Cases: GDPR and Data Governance: N day TTLs (30, 10, and 7 day policies are fairly standard for TTL)

You'll learn to achieve the following next:
1. Create a Table Partitioned by a given Column
2. Add or Remove Partitions for a given table
3. Modify an existing, non-partitioned table, to introduce partitioning (* requires some communication to your downstream data consumers)

In [9]:
# [Cell 38]
# if you need to go back and drop this table. 
# spark.sql("drop table workspace.dldg.nyc_taxi_by_day")

In [31]:
# [Cell 39]
# Table Creation with Partitions
(DeltaTable.createIfNotExists(spark)
  .tableName("workspace.dldg.nyc_taxi_by_day")
  .addColumn("VendorID", "BIGINT")
  .addColumn("tpep_pickup_datetime", "TIMESTAMP")
  .addColumn("tpep_dropoff_datetime", "TIMESTAMP", comment="trip drop off and partition source column")
  .addColumn("tpep_dropoff_date", DateType(), generatedAlwaysAs="CAST(tpep_dropoff_datetime AS DATE)")
  .addColumn("passenger_count", "DOUBLE")
  .addColumn("trip_distance", "DOUBLE")
  .addColumn("RatecodeID", "DOUBLE")
  .addColumn("store_and_fwd_flag", "STRING")
  .addColumn("PULocationID", "BIGINT")
  .addColumn("DOLocationID", "BIGINT")
  .addColumn("payment_type", "BIGINT")
  .addColumn("fare_amount", "DOUBLE")
  .addColumn("extra", "DOUBLE")
  .addColumn("mta_tax", "DOUBLE")
  .addColumn("tip_amount", "DOUBLE")
  .addColumn("tolls_amount", "DOUBLE")
  .addColumn("improvement_surcharge", "DOUBLE")
  .addColumn("total_amount", "DOUBLE")
  .addColumn("congestion_surcharge", "DOUBLE")
  .addColumn("airport_fee", "DOUBLE")
  .partitionedBy("tpep_dropoff_date")
  .property("description", "partitioned by taxi trip end time")
  .property("delta.logRetentionDuration", "interval 30 days")
  .property("delta.deletedFileRetentionDuration", "interval 1 day")
  .property("delta.dataSkippingNumIndexedCols", "10")
  .property("delta.checkpoint.writeStatsAsStruct", "true")
  .property("delta.checkpoint.writeStatsAsJson", "false")
  .execute()
)

In [32]:
# [Cell 40]
# read the data from our workspace.dldg.nyc_taxi table into the newly created nyc_taxi_by_day table 
# this will allow us to generate a partitioned table
# note: append will continue to add to the table, if you run this block multiple times - you'll have duplicates
(
    spark
    .table("workspace.dldg.nyc_taxi")
    .write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "false")
    .saveAsTable("workspace.dldg.nyc_taxi_by_day")
)

In [33]:
# [Cell 41]
spark.sql("describe extended workspace.dldg.nyc_taxi_by_day").show(30, truncate=False)

+----------------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------+-----------------------------------------+
|col_name                    |data_type                                                                                                                                                      |comment                                  |
+----------------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------+-----------------------------------------+
|VendorID                    |bigint                                                                                                                                                         |NULL                                     |
|tpep_pickup_datetime        |timestamp                             

In [34]:
# [Cell 42]
# removing bad data.
# ls -la ch05/spark-warehouse/nyc_taxi_by_day/
# will create some interesting table partitions: 
# tpep_dropoff_date=2009-01-01
# tpep_dropoff_date=2022-10-24
# tpep_dropoff_date=2022-10-25
# tpep_dropoff_date=2022-12-31

# we can investigate what is incorrect with the data in each of these partitions, or just wipe them out, as an exploration, 
# feel free to put your detectives hat on and see what is wrong with the data, or just delete the specific dates.
(
    DeltaTable
    .forName(spark, 'workspace.dldg.nyc_taxi_by_day')
    .delete(col("tpep_dropoff_date") < "2023-01-01")
)

DataFrame[num_affected_rows: bigint]

*[Cell 43]*

# Ensure the erroneous data is out of the current table Snapshot
While we have deleted the bad partitions, we haven't wiped them out fully, to do so we will need to vacuum the table which we'll get to later.

In [36]:
# [Cell 44]
delta_table = spark.table("workspace.dldg.nyc_taxi_by_day").where(col("tpep_dropoff_date") < "2023-01-01")

In [37]:
# [Cell 45]
delta_table.count()

0

In [38]:
# [Cell 46]
# the table itself has around 3 million trips
spark.table("workspace.dldg.nyc_taxi_by_day").count()

3066741

In [ ]:
# [Cell 47]
# Using time travel to 
dt = DeltaTable.forName(spark, 'workspace.dldg.nyc_taxi_by_day')
#dt.detail().printSchema()
"""
root
 |-- format: string (nullable = true)
 |-- id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- description: string (nullable = true)
 |-- location: string (nullable = true)
 |-- createdAt: timestamp (nullable = true)
 |-- lastModified: timestamp (nullable = true)
 |-- partitionColumns: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- numFiles: long (nullable = true)
 |-- sizeInBytes: long (nullable = true)
 |-- properties: map (nullable = true)
 |    |-- key: string
 |    |-- value: string (valueContainsNull = true)
 |-- minReaderVersion: integer (nullable = true)
 |-- minWriterVersion: integer (nullable = true)
 |-- tableFeatures: array (nullable = true)
 |    |-- element: string (containsNull = true)
"""

# view the table details
dt.detail().select("partitionColumns", "numFiles", "lastModified").show(truncate=False)

# describe the extended history of the table
spark.sql("describe extended workspace.dldg.nyc_taxi_by_day").show(40, truncate=True)

# gather partition days
from pyspark.sql.functions import desc
(spark
   .table('workspace.dldg.nyc_taxi_by_day')
   .select("tpep_dropoff_date")
   .distinct()
   .sort(desc("tpep_dropoff_date"))
   .show(33))

+-------------------+--------+-------------------+
|partitionColumns   |numFiles|lastModified       |
+-------------------+--------+-------------------+
|[tpep_dropoff_date]|33      |2026-10-25 02:13:42|
+-------------------+--------+-------------------+

+--------------------+--------------------+--------------------+
|            col_name|           data_type|             comment|
+--------------------+--------------------+--------------------+
|            VendorID|              bigint|                NULL|
|tpep_pickup_datetime|           timestamp|                NULL|
|tpep_dropoff_date...|           timestamp|trip drop off and...|
|   tpep_dropoff_date|                date|                NULL|
|     passenger_count|              double|                NULL|
|       trip_distance|              double|                NULL|
|          RatecodeID|              double|                NULL|
|  store_and_fwd_flag|              string|                NULL|
|        PULocationID|       

*[Cell 48]*

# Recovering Data using Partitions using "Replays"
> Let's face it. Even with the best intentions in place, we are all human and make mistakes. In your career as a data engineer, one thing you'll be required to learn is the art of data recovery. When we recover data, the process is commonly called 'replaying' since the action we are taking is to rollback the clock, or rewind, to an earlier point in time. This enables us to remove problematic changes to a table, and replace the erroneous data with whatever the "fixed" data is.






In [1]:
# [Cell 49]
# using write mode 'overwrite' and replaceWhere condition
# example values: replay the 2023-01-01 trips from the source table into the partitioned table
recovery_table = "workspace.dldg.nyc_taxi"
partition_col = "tpep_dropoff_date"
table_to_fix = "workspace.dldg.nyc_taxi_by_day"
(
  spark
    .table(recovery_table)
    # replaceWhere requires every written row to match the condition (DELTA_REPLACE_WHERE_MISMATCH);
    # tpep_dropoff_date is generated from tpep_dropoff_datetime, so filter on the source column
    .where(col("tpep_dropoff_datetime").cast("date") == "2023-01-01")
    # the source parquet has TIMESTAMP_NTZ columns; overwrite would try to change the target's TIMESTAMP
    # columns to match, which the generated column blocks (DELTA_GENERATED_COLUMNS_DATA_TYPE_MISMATCH)
    .withColumn("tpep_pickup_datetime", col("tpep_pickup_datetime").cast("timestamp"))
    .withColumn("tpep_dropoff_datetime", col("tpep_dropoff_datetime").cast("timestamp"))
    .write
    .format("delta")
    .mode("overwrite")
    .option("replaceWhere", f"{partition_col} == '2023-01-01'")
    .saveAsTable(table_to_fix)
)    


NameError: name 'spark' is not defined